In [29]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import itertools
import os
import random
import multiprocessing
import sys
import subprocess

BIO_LIP_COLUMNS = [
"PDB ID",
"Receptor chain",
"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
"Binding site number code",
"Ligand_ID",
"Ligand_chain",
"Ligand serial number",
"    Binding site residues (with PDB residue numbering)",
"    Binding site residues (with residue re-numbered starting from 1)",
"Catalytic site residues (different sites are separated by ';') (with PDB residue numbering)",
"    Catalytic site residues (different sites are separated by ';') (with residue re-numbered starting from 1)",
"EC number",
"GO terms",
"Binding affinity by manual survey of the original literature. The information in '()' is the PubMed ID",
"Binding affinity provided by the Binding MOAD database. The information in '()' is the ligand information in Binding MOAD",
"Binding affinity provided by the PDBbind-CN database. The information in '()' is the ligand information in PDBbind-CN",
"Binding affinity provided by the BindingDB database",
"UniProt ID",
"PubMed ID",
"Residue sequence number of the ligand (field _atom_site.auth_seq_id in PDBx/mmCIF format)",
"Receptor sequence"]



# now the invalid ligands are stored as strings, written one by one.
INVALID_LIGANDS = ['144', '15P', '1PE', '2F2', '2JC', '3HR', '3SY', '7N5', '7PE', '9JE', 'AAE', 'ABA', 'ACE', 'ACN', 'ACT', 'ACY', 'AZI', 'BAM', 'BCN', 'BCT', 'BDN', 'BEN', 'BME', 'BO3', 'BTB', 'BTC', 'BU1', 'C8E', 'CAD', 'CAQ', 'CBM', 'CCN', 'CIT', 'CL',
'CM', 'CMO', 'CO3', 'CPT', 'CXS', 'D10', 'DEP', 'DIO', 'DMS', 'DN', 'DOD', 'DOX', 'EDO', 'EEE', 'EGL', 'EOH', 'EOX', 'EPE', 'ETF', 'FCY', 'FJO', 'FLC', 'FMT', 'FW5', 'GOL', 'GSH', 'GTT', 'GYF', 'HED', 'IHP', 'IHS', 'IMD', 'IOD', 'IPA', 'IPH',
'LDA', 'MB3', 'MEG', 'MES', 'MLA', 'MLI', 'MOH', 'MPD', 'MRD', 'MSE', 'MYR', 'N', 'NA', 'NH2', 'NH4', 'NHE', 'NO3', 'O4B', 'OHE', 'OLA', 'OLC', 'OMB', 'OME', 'OXA', 'P6G', 'PE3', 'PE4', 'PEG', 'PEO', 'PEP', 'PG0', 'PG4', 'PGE', 'PGR',
'PLM', 'PO4', 'POL', 'POP', 'PVO', 'SAR', 'SCN', 'SEO', 'SEP', 'SIN', 'SO4', 'SPD', 'SPM', 'SR', 'STE', 'STO', 'STU', 'TAR', 'TBU', 'TME', 'TPO', 'TRS', 'UNK', 'UNL', 'UNX', 'UPL', 'URE']


# now with these: SO4, GOL, EDO, PO4, ACT, PEG, DMS, TRS, PGE, PG4, FMT, EPE, MPD, MES, CD, IOD
CRYSTALIZATION_LIGANDS = ['SO4', 'GOL', 'EDO', 'PO4', 'ACT', 'PEG', 'DMS', 'TRS', 'PGE', 'PG4', 'FMT', 'EPE', 'MPD', 'MES', 'CD', 'IOD']

ALL_INVALID_LIGANDS = INVALID_LIGANDS + CRYSTALIZATION_LIGANDS



df = pd.read_csv('/Users/jerometubiana/Downloads/BioLIP.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)
relevant_columns = ["PDB ID", "Receptor chain", "Ligand_ID", "Ligand_chain",'Receptor sequence']
df = df[relevant_columns].astype('str')
print(df.shape)
df = df[~df['Ligand_ID'].str.contains('DNA|RNA|PEPTIDE|NONE|nan', case=False, na=False)]
print(df.shape)
df = df[~df['Ligand_ID'].map(lambda x: x in ALL_INVALID_LIGANDS)]
print(df.shape)


# df = df[:1000:10]
df = df.reset_index(drop=True)

/var/folders/52/fmf_vkm95_s5lgbwbnzb1yb80000gn/T/ipykernel_2237/1576982736.py:51: DtypeWarning: Columns (9,10,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/jerometubiana/Downloads/BioLIP.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)


(855312, 5)
(626445, 5)
(610590, 5)


In [30]:

def cluster_sequences(list_sequences, seqid=1.0, coverage=0.8, covmode='0', path2mmseqstmp='/Users/jerometubiana/tmp/',
                      path2mmseqs='/opt/anaconda3/bin/mmseqs',threads=8
                      ):

    rng = np.random.randint(0, high=int(1e6))
    tmp_input = os.path.join(path2mmseqstmp, 'tmp_input_file_%s.fasta' % rng)
    tmp_output = os.path.join(path2mmseqstmp, 'tmp_output_file_%s' % rng)

    with open(tmp_input, 'w') as f:
        for k, sequence in enumerate(list_sequences):
            f.write('>%s\n' % k)
            f.write('%s\n' % sequence)

    command = ('{mmseqs} easy-cluster {fasta} {result} {tmp} --threads {threads} --min-seq-id %s -c %s --cov-mode %s' % (
        seqid, coverage, covmode)).format(threads=threads, mmseqs=path2mmseqs, fasta=tmp_input, result=tmp_output, tmp=path2mmseqstmp)
    subprocess.run(command.split(' '))

    with open(tmp_output + '_rep_seq.fasta', 'r') as f:
        representative_indices = [int(x[1:-1]) for x in f.readlines()[::2]]
    cluster_indices = np.zeros(len(list_sequences), dtype=int)
    table = pd.read_csv(tmp_output + '_cluster.tsv', sep='\t', header=None).to_numpy(dtype=int)
    for i, j in table:
        if i in representative_indices:
            cluster_indices[j] = representative_indices.index(i)
    for file in [tmp_output + '_rep_seq.fasta', tmp_output + '_all_seqs.fasta', tmp_output + '_cluster.tsv']:
        os.remove(file)
    return np.array(cluster_indices), np.array(representative_indices)


cluster_indices, representative_indices = cluster_sequences(df['Receptor sequence'].to_list(), seqid=0.7,coverage=0.7)

df['cluster_indices'] = cluster_indices
all_ligands = df.groupby('cluster_indices')['Ligand_ID'].unique().map(lambda x: '|'.join(x))
df['Ligand_all']=all_ligands[df['cluster_indices']].reset_index(drop=True)

df_nr = df.iloc[representative_indices].reset_index(drop=True)
print(df.shape, df_nr.shape)

easy-cluster /Users/jerometubiana/tmp/tmp_input_file_738132.fasta /Users/jerometubiana/tmp/tmp_output_file_738132 /Users/jerometubiana/tmp/ --threads 8 --min-seq-id 0.7 -c 0.7 --cov-mode 0 

MMseqs Version:                     	14.7e284
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.7
Coverage mode                       	0
Compositional bias                  	1
Compositional bias                  	1
Diagonal scoring                    	true
Exact k-mer matching   

In [ ]:
df_nr = df.iloc[representative_indices].reset_index(drop=True)
df_nr = df_nr.drop(columns=['Receptor sequence','Ligand_chain']).rename(columns={'PDB ID':'tar_protein',
                                                                         'Receptor chain':'tar_chain',
                                                                         'Ligand_ID':'ligand',
                                                                         'Ligand_all':'ligand_all'})
df_nr['tar_motif'] = [None for _ in range(len(df_nr))]
df_nr = df_nr[ ['ligand','tar_protein','tar_chain','tar_motif','ligand_all'] ]
df_nr.to_csv('../example_inputs/biolip2_nr_database.csv',index=False)
# df_nr.to_csv('../example_inputs/biolip2_nr_mini_database.csv')